In [12]:

!pip install bcrypt --quiet
import hashlib
import bcrypt
import secrets
import itertools
import string
import time
import hmac
import struct
import datetime

All imports successful. Ready to run the demo.


In [20]:
def hash_with_md5(password):
    return hashlib.md5(password.encode('utf-8')).hexdigest()
def hash_with_sha256(password):
    return hashlib.sha256(password.encode('utf-8')).hexdigest()
def hash_with_bcrypt(password, cost=10):
    password_as_bytes = password.encode('utf-8')
    salt = bcrypt.gensalt(rounds=cost)
    hashed = bcrypt.hashpw(password_as_bytes, salt)
    return hashed.decode('utf-8')
def verify_bcrypt_hash(password, stored_hash):
    return bcrypt.checkpw(password.encode('utf-8'), stored_hash.encode('utf-8'))


test_passwords = ["hello", "123456", "P@ssw0rd!", "hello"]
for password in test_passwords:
    md5_result    = hash_with_md5(password)
    sha256_result = hash_with_sha256(password)[:32]
    print(f"{password:<15} {md5_result:<35} {sha256_result}")

print("bcrypt hashes ")
print(f"  Hash 1: {hash_with_bcrypt('hello')}")
print(f"  Hash 2: {hash_with_bcrypt('hello')}")
print(f"  Hash 3: {hash_with_bcrypt('hello')}")


hello           5d41402abc4b2a76b9719d911017c592    2cf24dba5fb0a30e26e83b2ac5b9e29e
123456          e10adc3949ba59abbe56e057f20f883e    8d969eef6ecad3c29a3a629280e686cf
P@ssw0rd!       8a24367a1f46c141048752f2d5bbd14b    0e44ce7308af2b3de5232e4616403ce7
hello           5d41402abc4b2a76b9719d911017c592    2cf24dba5fb0a30e26e83b2ac5b9e29e
bcrypt hashes 
  Hash 1: $2b$10$IF/QFtYBCJXzN5efbazlm.Kq9Gz0avSiuXHJlniFEp4F7MaAYeX4i
  Hash 2: $2b$10$uqvKPOiaUMEmVx3nLpLXAOfKODedF7vMS7/3kX9iBxl1sF2lnjt2q
  Hash 3: $2b$10$6CzntPJZ98ImYRsVepldBOnW.ZGbEiDslQv8a33oJZuYvnws0edtu


In [21]:
number_of_hashes = 100000
start_time = time.time()
for i in range(number_of_hashes):
    hash_with_md5("testpassword")
end_time = time.time()
time_taken_md5 = end_time - start_time
md5_speed = number_of_hashes / time_taken_md5

start_time = time.time()
for i in range(number_of_hashes):
    hash_with_sha256("testpassword")
end_time = time.time()
time_taken_sha256 = end_time - start_time
sha256_speed = number_of_hashes / time_taken_sha256


number_of_bcrypt = 5
start_time = time.time()
for i in range(number_of_bcrypt):
    hash_with_bcrypt("testpassword", cost=10)
end_time = time.time()
time_taken_bcrypt = end_time - start_time
bcrypt_speed = number_of_bcrypt / time_taken_bcrypt

print(f"  MD5    : {md5_speed:>15,.0f} hashes per second")
print(f"  SHA-256: {sha256_speed:>15,.0f} hashes per second")
print(f"  bcrypt : {bcrypt_speed:>15.2f} hashes per second ")

paper_md5_speed    = 41_000_000
paper_sha256_speed = 26_000_000
paper_bcrypt_speed = 27
charset_size        = 69
six_char_combinations = charset_size ** 6

time_md5    = six_char_combinations / paper_md5_speed
time_sha256 = six_char_combinations / paper_sha256_speed
time_bcrypt = six_char_combinations / paper_bcrypt_speed

print(f"    Total combinations (6 chars, 69-char set): {six_char_combinations:,}")
print(f"    With MD5    : {time_md5:.0f} seconds = {time_md5/60:.0f} minutes")
print(f"    With SHA-256: {time_sha256:.0f} seconds = {time_sha256/60:.0f} minutes")
print(f"    With bcrypt : {time_bcrypt:.0f} seconds = {time_bcrypt/86400/365:.0f} years")

  MD5    :       1,117,578 hashes per second
  SHA-256:         953,956 hashes per second
  bcrypt :           11.77 hashes per second 
    Total combinations (6 chars, 69-char set): 107,918,163,081
    With MD5    : 2632 seconds = 44 minutes
    With SHA-256: 4151 seconds = 69 minutes
    With bcrypt : 3996969003 seconds = 127 years


In [23]:
def brute_force_attack(target_hash, charset, max_length=4, time_limit_seconds=10):

    start_time   = time.time()
    total_attempts = 0

    for current_length in range(1, max_length + 1):
        combinations_at_this_length = len(charset) ** current_length
        print(f"  Trying length {current_length} ({combinations_at_this_length:,} combinations)")


        for character_tuple in itertools.product(charset, repeat=current_length):
            guess = ''.join(character_tuple)
            total_attempts += 1

            if hash_with_md5(guess) == target_hash:
                elapsed = time.time() - start_time
                print(f"  CRACKED: '{guess}' {elapsed:.3f} seconds")
                return guess

digit_charset = string.digits


for target_password in ["7", "42"]:
    target_hash = hash_with_md5(target_password)
    print(f"Target: '{target_password}'   Hash: {target_hash}")
    brute_force_attack(target_hash, digit_charset, max_length=5)
    print()

Target: '7'   Hash: 8f14e45fceea167a5a36dedd4bea2543
  Trying length 1 (10 combinations)
  CRACKED: '7' 0.000 seconds

Target: '42'   Hash: a1d0c6e83f027327d8461063f4ac58a6
  Trying length 1 (10 combinations)
  Trying length 2 (100 combinations)
  CRACKED: '42' 0.000 seconds



In [25]:
dictionary_small = [
    "password", "123456", "qwerty", "dragon", "iloveyou",
    "admin", "111111", "hello", "sunshine", "abc123",
    "letmein", "monkey", "master", "batman", "123123"
]

dictionary_extended = dictionary_small + [
    "Qwerty123", "P@ssw0rd!", "vbjdsubv", "AaBbCc",
    "R14ERe", "password1", "password123", "football",
    "superman", "trustno1", "login", "welcome", "guest"
]


def dictionary_attack(target_hash, dictionary, dictionary_name):

    start_time = time.time()

    for position, word in enumerate(dictionary):
        if hash_with_md5(word) == target_hash:
            elapsed = time.time() - start_time
            print(f"  [{dictionary_name}] CRACKED: '{word}' at position {position+1} in {elapsed:.4f}s")
            return word

    elapsed = time.time() - start_time
    print(f"  [{dictionary_name}] Not found ({len(dictionary)} words checked in {elapsed:.4f}s)")
    return None



test_cases = [
    ("123456"),
    ("dragon"),
    ("Qwerty123"),
    ("P@ssw0rd!"),
    ("xK9#mQ2p"),
]

for password in test_cases:
    target_hash = hash_with_md5(password)
    print(f"Password: '{password}'")

    dictionary_attack(target_hash, dictionary_small,    "Dict 1")
    dictionary_attack(target_hash, dictionary_extended, "Dict 2")
    print()


Password: '123456'
  [Dict 1] CRACKED: '123456' at position 2 in 0.0000s
  [Dict 2] CRACKED: '123456' at position 2 in 0.0000s

Password: 'dragon'
  [Dict 1] CRACKED: 'dragon' at position 4 in 0.0000s
  [Dict 2] CRACKED: 'dragon' at position 4 in 0.0000s

Password: 'Qwerty123'
  [Dict 1] Not found (15 words checked in 0.0000s)
  [Dict 2] CRACKED: 'Qwerty123' at position 16 in 0.0000s

Password: 'P@ssw0rd!'
  [Dict 1] Not found (15 words checked in 0.0000s)
  [Dict 2] CRACKED: 'P@ssw0rd!' at position 17 in 0.0000s

Password: 'xK9#mQ2p'
  [Dict 1] Not found (15 words checked in 0.0000s)
  [Dict 2] Not found (28 words checked in 0.0000s)



In [26]:

common_passwords = [
    "123456", "password", "qwerty", "dragon", "iloveyou",
    "admin", "hello", "sunshine", "111111", "123123",
    "password1", "abc123", "monkey", "master", "letmein"
]
rainbow_table = {}
for password in common_passwords:
    hashed = hash_with_md5(password)
    rainbow_table[hashed] = password

attack_targets = ["123456", "dragon", "iloveyou", "xK9#mQ2p", "P@ssw0rd!"]
for target_password in attack_targets:
    target_hash = hash_with_md5(target_password)
    result      = rainbow_table.get(target_hash)
    if result:
        print(f"  '{target_password}' -> CRACKED instantly (found '{result}' in table)")
    else:
        print(f"  '{target_password}' -> Not in table")


shared_password = "123456"

print(f"WITHOUT salting ")
print(f"  User Alice: {hash_with_md5(shared_password)}")
print(f"  User Bob  : {hash_with_md5(shared_password)}")
print(f"  User Carol: {hash_with_md5(shared_password)}")


print(f"WITH salting ")

def hash_with_manual_salt(password, salt=None):

    if salt is None:
        salt = secrets.token_hex(8)      # 8 random bytes = 16 hex characters
    salted_input = salt + password       # salt is prepended to the password
    hashed       = hash_with_md5(salted_input)
    return hashed, salt

for user_name in ["Alice", "Bob", "Carol"]:
    user_hash, user_salt = hash_with_manual_salt(shared_password)
    print(f"  {user_name}: hash={user_hash}   salt={user_salt}")


  '123456' -> CRACKED instantly (found '123456' in table)
  'dragon' -> CRACKED instantly (found 'dragon' in table)
  'iloveyou' -> CRACKED instantly (found 'iloveyou' in table)
  'xK9#mQ2p' -> Not in table
  'P@ssw0rd!' -> Not in table
WITHOUT salting 
  User Alice: e10adc3949ba59abbe56e057f20f883e
  User Bob  : e10adc3949ba59abbe56e057f20f883e
  User Carol: e10adc3949ba59abbe56e057f20f883e
WITH salting 
  Alice: hash=3181faf842c2ba1c02a37e85244f3375   salt=84f52f9193ea6e6b
  Bob: hash=d155da4bfcaac1b630f54879a32cdab3   salt=7fc5d36ad39b2960
  Carol: hash=6244f101e3064bd41f0ccf2e0f5cf4ef   salt=54cf2481d70520f5


In [31]:
def generate_totp_code(shared_secret, time_window_seconds=30, code_length=6):


    current_time_counter = int(time.time()) // time_window_seconds

    counter_as_bytes = struct.pack('>Q', current_time_counter)
    mac = hmac.new(
        shared_secret.encode('utf-8'),
        counter_as_bytes,
        hashlib.sha1
    ).digest()

    offset      = mac[-1] & 0x0F
    four_bytes  = struct.unpack('>I', mac[offset : offset + 4])[0]
    truncated   = four_bytes & 0x7FFFFFFF


    code = truncated % (10 ** code_length)
    return str(code).zfill(code_length)



SHARED_SECRET = "DEMO_SECRET_KEY_12345"


server_code = generate_totp_code(SHARED_SECRET)
phone_code  = generate_totp_code(SHARED_SECRET)

print(f"  Code computed by SERVER : {server_code}")
print(f"  Code computed by PHONE  : {phone_code}")
print(f"  match")


stored_password_hash = hash_with_bcrypt("mypassword", cost=4)

def mfa_login_attempt(password_attempt, otp_attempt):

    password_is_correct = verify_bcrypt_hash(password_attempt, stored_password_hash)


    current_valid_otp   = generate_totp_code(SHARED_SECRET)
    otp_is_correct      = (otp_attempt == current_valid_otp)

    print(f" {'CORRECT' if otp_is_correct else 'WRONG'}"
          f" ", end="")

    if password_is_correct and otp_is_correct:
        print("ACCESS GRANTED")
    elif password_is_correct and not otp_is_correct:
        print("DENIED — correct password but wrong OTP")
    elif not password_is_correct and otp_is_correct:
        print("DENIED — wrong password")
    else:
        print("DENIED — both wrong")


current_otp = generate_totp_code(SHARED_SECRET)

print("correct password AND correct OTP")
mfa_login_attempt("mypassword", current_otp)

print("Attacker who cracked the password hash but has no device:")
mfa_login_attempt("mypassword", "000000")
print()
print("Someone with the OTP device but wrong password:")
mfa_login_attempt("wrongpassword", current_otp)
print()
print("Everything wrong:")
mfa_login_attempt("wrongpassword", "000000")

  Code computed by SERVER : 150017
  Code computed by PHONE  : 150017
  match
correct password AND correct OTP
 CORRECT ACCESS GRANTED
Attacker who cracked the password hash but has no device:
 WRONG DENIED — correct password but wrong OTP

Someone with the OTP device but wrong password:
 CORRECT DENIED — wrong password

Everything wrong:
 WRONG DENIED — both wrong


In [32]:

def format_time(seconds):
    if seconds < 1:             return "< 1 second"
    if seconds < 60:            return f"{seconds:.0f} seconds"
    if seconds < 3600:          return f"{seconds/60:.0f} minutes"
    if seconds < 86400:         return f"{seconds/3600:.1f} hours"
    if seconds < 86400 * 365:   return f"{seconds/86400:.0f} days"
    return                             f"{seconds/86400/365:.0f} years"



paper_md5_speed    = 41_000_000
paper_sha256_speed = 26_000_000
paper_bcrypt_speed = 27

charset_size = 69

print("PAPER RESULTS: Time to Brute Force by Password Length")
print(f"  {'Length':<8} {'Combinations':<20} {'MD5':<14} {'SHA-256':<14} {'bcrypt'}")

for password_length in range(1, 10):
    total_combinations = charset_size ** password_length
    time_md5    = total_combinations / paper_md5_speed
    time_sha256 = total_combinations / paper_sha256_speed
    time_bcrypt = total_combinations / paper_bcrypt_speed
    print(f"  {password_length:<8} {total_combinations:<20,} "
          f"{format_time(time_md5):<14} "
          f"{format_time(time_sha256):<14} "
          f"{format_time(time_bcrypt)}")



PAPER RESULTS: Time to Brute Force by Password Length
  Length   Combinations         MD5            SHA-256        bcrypt
  1        69                   < 1 second     < 1 second     3 seconds
  2        4,761                < 1 second     < 1 second     3 minutes
  3        328,509              < 1 second     < 1 second     3.4 hours
  4        22,667,121           < 1 second     < 1 second     10 days
  5        1,564,031,349        38 seconds     1 minutes      2 years
  6        107,918,163,081      44 minutes     1.2 hours      127 years
  7        7,446,353,252,589    2 days         3 days         8745 years
  8        513,798,374,428,641  145 days       229 days       603424 years
  9        35,452,087,835,576,229 27 years       43 years       41636234 years
